# Kurgans UNet Experiments on Kaggle

Notebook only prepares Kaggle paths, launches `02_unet_segmentation/run_kaggle_experiments.sh`, previews outputs, and archives runs.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

KAGGLE_INPUT = Path('/kaggle/input')
WORK_REPO = Path('/kaggle/working/Geodata_Archaeology_CV')
DATA_ROOT = Path('/kaggle/input/datasets/matanerdy/kurgans-dataset/kurgans_dataset')
SEG_DIR = WORK_REPO / '02_unet_segmentation'
RUN_ROOT = SEG_DIR / 'runs'
BASELINE_DIR = RUN_ROOT / 'baseline_all_modalities_ce_dice'

print('Kaggle input:', KAGGLE_INPUT)
print('Work repo:', WORK_REPO)
print('Data root:', DATA_ROOT)

## GPU Check

In [ ]:
!nvidia-smi || true

import torch

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))

## Find And Copy Repository

In [ ]:
def find_repo_root(input_root: Path) -> Path:
    matches = []
    for path in input_root.rglob('02_unet_segmentation'):
        if path.is_dir() and (path / 'train.py').exists():
            matches.append(path.parent)
    if not matches:
        raise FileNotFoundError(
            'Could not find a Kaggle input repository containing 02_unet_segmentation/train.py'
        )
    matches = sorted(matches, key=lambda p: (len(p.parts), str(p)))
    return matches[0]

repo_input = find_repo_root(KAGGLE_INPUT)
print('Found repository input:', repo_input)

if WORK_REPO.exists():
    shutil.rmtree(WORK_REPO)

def ignore_patterns(directory, names):
    ignored = {'.git', '.DS_Store', '__pycache__', 'runs'} & set(names)
    if Path(directory).name == 'datasets' and 'kurgans_dataset' in names:
        ignored.add('kurgans_dataset')
    return ignored

shutil.copytree(repo_input, WORK_REPO, ignore=ignore_patterns)
print('Copied repository to:', WORK_REPO)
print('Segmentation directory exists:', SEG_DIR.exists())

## Dataset Check

In [ ]:
required = [
    DATA_ROOT,
    DATA_ROOT / 'metadata.csv',
    DATA_ROOT / 'images',
    DATA_ROOT / 'masks',
    SEG_DIR / 'run_kaggle_experiments.sh',
]
for path in required:
    print(path, 'OK' if path.exists() else 'MISSING')
    if not path.exists():
        raise FileNotFoundError(path)

print('Images:', len(list((DATA_ROOT / 'images').glob('*.npy'))))
print('Masks:', len(list((DATA_ROOT / 'masks').glob('*.npy'))))

## Run Experiments

In [ ]:
env = os.environ.copy()
env['DATA_ROOT'] = str(DATA_ROOT)
env['RUN_ROOT'] = str(RUN_ROOT)

subprocess.run(
    ['bash', 'run_kaggle_experiments.sh'],
    cwd=SEG_DIR,
    env=env,
    check=True,
)

## History

In [ ]:
import pandas as pd

history_path = BASELINE_DIR / 'history.csv'
history = pd.read_csv(history_path)
display(history.tail())

## Prediction Examples

In [ ]:
from IPython.display import Image, display

prediction_path = BASELINE_DIR / 'prediction_examples.png'
if not prediction_path.exists():
    prediction_path = BASELINE_DIR / 'prediction_examples_eval.png'

display(Image(filename=str(prediction_path)))

## Archive Runs

In [ ]:
from IPython.display import FileLink, display

archive_base = Path('/kaggle/working/kurgans_runs')
archive_path = Path(
    shutil.make_archive(
        str(archive_base),
        'zip',
        root_dir=RUN_ROOT.parent,
        base_dir=RUN_ROOT.name,
    )
)
print('Created archive:', archive_path)
display(FileLink(str(archive_path)))